In [ ]:
import pandas as pd
import os

In [ ]:
# Todos os indicadores de educação são elaborados pelo Inep
# df: IGC
# df_ideb : IDEB
# df_ide_total: IDE
# df_afd: AFD

# IGC (faixa)

In [ ]:
# índice geral de cursos
IGC = pd.read_excel('IGC.xlsx')

In [ ]:
IGC.shape

(22855, 4)

In [ ]:
IGC.head()

,Ano,Nome da IES,UF,IGC (Faixa)
0,2012,UNIVERSIDADE FEDERAL DE MATO GROSSO,MT,4
1,2012,UNIVERSIDADE DE BRASÍLIA,DF,4
2,2012,UNIVERSIDADE FEDERAL DE SERGIPE,SE,4
3,2012,UNIVERSIDADE FEDERAL DO AMAZONAS,AM,3
4,2012,UNIVERSIDADE FEDERAL DO PIAUÍ,PI,3


In [ ]:
IGC['IGC (Faixa)']

,IGC (Faixa)
0,4
1,4
2,4
3,3
4,3
...,...
22850,4
22851,4
22852,4
22853,4


In [ ]:
# Converte para numérico, substituindo valores não numéricos por NaN
IGC['IGC (Faixa)'] = pd.to_numeric(IGC['IGC (Faixa)'], errors='coerce')

# Remove linhas com NaN na coluna 'IGC (Faixa)'
IGC = IGC.dropna(subset=['IGC (Faixa)'])

# Opcional: converter para inteiro após remoção
IGC['IGC (Faixa)'] = IGC['IGC (Faixa)'].astype(int)

In [ ]:
# Agrupar por Ano e UF e contar frequência de cada valor de IGC (Faixa)
freq = IGC.groupby(['Ano', 'UF', 'IGC (Faixa)']).size().unstack(fill_value=0)

In [ ]:
print(freq.index.get_level_values('Ano').unique())

Index([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023], dtype='int64', name='Ano')


In [ ]:
freq_weighted = freq.copy()

In [ ]:
pesos = [1, 2, 3, 4, 5]

freq_weighted = freq.copy()

for peso_str, col in zip(pesos, freq.columns):
    freq_weighted[col] = freq[col] * peso_str

In [ ]:
# Calcula a soma ponderada
soma_ponderada = freq_weighted.sum(axis=1)

In [ ]:
# Calcula a soma das frequências
soma_frequencia = freq.sum(axis=1)

In [ ]:
# Divide para obter a média ponderada
media_ponderada = soma_ponderada / soma_frequencia

In [ ]:
# Para obter o resultado de forma organizada, pode transformar em DataFrame
resultado = media_ponderada.reset_index()
resultado.columns = ['Ano', 'UF', 'Media_Ponderada']

In [ ]:
resultado.head()

,Ano,UF,Media_Ponderada
0,2012,AC,2.625000
1,2012,AL,2.500000
2,2012,AM,2.789474
3,2012,AP,2.500000
4,2012,BA,2.831461


# ideb

In [ ]:
import pandas as pd

# Carregar a planilha IDEB.xlsx
df_ideb = pd.read_excel('IDEB.xlsx')

# Exibir o DataFrame
print(df_ideb)

       ano sigla_uf      rede       ensino  anos_escolares  taxa_aprovacao  \
0     2021       AC     total        medio     todos (1-4)            86.4   
1     2021       DF  estadual        medio     todos (1-4)            84.8   
2     2021       AM     total        medio     todos (1-4)            91.6   
3     2005       BA   privada        medio     todos (1-4)            90.0   
4     2021       RR     total        medio     todos (1-4)            86.7   
...    ...      ...       ...          ...             ...             ...   
2668  2013       DF   privada  fundamental  iniciais (1-5)            99.3   
2669  2019       DF   privada  fundamental  iniciais (1-5)            99.7   
2670  2019       SC   privada  fundamental  iniciais (1-5)            99.4   
2671  2015       MG   privada  fundamental  iniciais (1-5)            99.0   
2672  2019       PR   privada  fundamental  iniciais (1-5)            98.9   

      indicador_rendimento  nota_saeb_matematica  nota_saeb_lin

In [ ]:
df_ideb.columns

Index(['ano', 'sigla_uf', 'rede', 'ensino', 'anos_escolares', 'taxa_aprovacao',
       'indicador_rendimento', 'nota_saeb_matematica',
       'nota_saeb_lingua_portuguesa', 'nota_saeb_media_padronizada', 'ideb'],
      dtype='object')

In [ ]:
# Agrupar por sigla_uf e ano e calcular a média do IDEB
media_ideb = df_ideb.groupby(['sigla_uf', 'ano'])['ideb'].mean().reset_index()

# Renomear a coluna de média se desejar
media_ideb = media_ideb.rename(columns={'ideb': 'Media_IDEB'})


In [ ]:
media_ideb

,sigla_uf,ano,Media_IDEB
0,AC,2005,3.900000
1,AC,2007,4.163636
2,AC,2009,4.037500
3,AC,2011,4.527273
4,AC,2013,4.800000
...,...,...,...
238,TO,2013,4.645455
239,TO,2015,4.709091
240,TO,2017,5.209091
241,TO,2019,5.254545


# IED

In [ ]:
df_ied = pd.read_excel('IED TOTAL.xlsx')

In [ ]:
df_ied.head()

,Ano,UF do IES,Nível 1,Nível 6,Rede,Zona
0,2013,RO,100,0,Estadual,Rural
1,2013,RO,100,0,Municipal,Rural
2,2013,RO,100,0,Municipal,Rural
3,2013,RO,20,0,Municipal,Rural
4,2013,RO,100,0,Municipal,Rural


In [ ]:
df_ied.columns = df_ied.columns.str.strip()

In [ ]:
df_ied['Nível 1'] = pd.to_numeric(df_ied['Nível 1'], errors='coerce')
df_ied['Nível 6'] = pd.to_numeric(df_ied['Nível 6'], errors='coerce')

In [ ]:
# Criar a nova coluna 'Nível IDE' como a diferença entre 'Nível 1' e 'Nível 6'
df_ied['Nível IDE'] = df_ied['Nível 1'] - df_ied['Nível 6']

In [ ]:
# Agrupar por 'Ano' e 'UF do IES' e calcular a média
media_nivel_ide = df_ied.groupby(['Ano', 'UF do IES'])['Nível IDE'].mean().reset_index()

In [ ]:
print(media_nivel_ide)

      Ano UF do IES  Nível IDE
0    2013        AC  56.129086
1    2013        AL  36.638196
2    2013        AM  44.677749
3    2013        AP  48.476338
4    2013        BA  36.122779
..    ...       ...        ...
184  2019        RS  16.324910
185  2019        SC   8.452320
186  2019        SE  33.715358
187  2019        SP  13.675325
188  2019        TO  36.080990

[189 rows x 3 columns]


# AFD

In [ ]:
df_afd = pd.read_excel('AFD.xlsx')

In [ ]:
df_afd.columns

Index(['Ano', 'UF', 'Grupo 1 Fundamental 5º e 9º', 'Grupo 1 Médio 3º ano ',
       'Zona', 'Ensino'],
      dtype='object')

In [ ]:
df_afd['Grupo 1 Fundamental 5º e 9º'] = pd.to_numeric(df_afd['Grupo 1 Fundamental 5º e 9º'], errors='coerce')
df_afd['Grupo 1 Médio 3º ano '] = pd.to_numeric(df_afd['Grupo 1 Médio 3º ano '], errors='coerce')

In [ ]:
# Agrupar por 'ano' e 'sigla_uf' e calcular a média para as colunas específicas
media_grupos = df_afd.groupby(['Ano', 'UF'])[['Grupo 1 Fundamental 5º e 9º', 'Grupo 1 Médio 3º ano ']].mean().reset_index()

# Renomear as colunas das médias se desejar
media_grupos = media_grupos.rename(columns={
    'Grupo 1 Fundamental 5º e 9º': 'Media Fundamental 5º e 9º',
    'Grupo 1 Médio 3º ano ': 'Media Médio 3º ano'
})

print(media_grupos)

      Ano  UF  Media Fundamental 5º e 9º  Media Médio 3º ano
0    2013  AC                  15.833823           33.815758
1    2013  AL                  27.289150           47.455682
2    2013  AM                  21.917828           56.723291
3    2013  AP                  28.557887           70.050806
4    2013  BA                  21.064921           31.996504
..    ...  ..                        ...                 ...
130  2017  RS                  52.950960           60.719814
131  2017  SC                  67.637449           63.652429
132  2017  SE                  48.143864           68.306944
133  2017  SP                  69.826962           63.453375
134  2017  TO                  39.366492           41.376471

[135 rows x 4 columns]


# calculando Phi

In [ ]:
# Renomear colunas para consistência
media_ideb = media_ideb.rename(columns={'ano': 'Ano', 'sigla_uf': 'UF'})
media_nivel_ide = media_nivel_ide.rename(columns={'UF do IES': 'UF'})
media_grupos = media_grupos.rename(columns={
    'Media Fundamental 5º e 9º': 'Media_Fundamental',
    'Media Médio 3º ano': 'Media_Medio'
})

In [ ]:
print(resultado.columns)
print(media_ideb.columns)
print(media_nivel_ide.columns)
print(media_grupos.columns)

Index(['Ano', 'UF', 'Media_Ponderada'], dtype='object')
Index(['UF', 'Ano', 'Media_IDEB'], dtype='object')
Index(['Ano', 'UF', 'Nível IDE'], dtype='object')
Index(['Ano', 'UF', 'Media_Fundamental', 'Media_Medio'], dtype='object')


In [ ]:
# Remova espaços extras
for df in [resultado, media_ideb, media_nivel_ide, media_grupos]:
    df.columns = df.columns.str.strip()

In [ ]:
# Confirme os tipos de dados
for df in [resultado, media_ideb, media_nivel_ide, media_grupos]:
    print(df[['Ano', 'UF']].dtypes)

Ano     int64
UF     object
dtype: object
Ano     int64
UF     object
dtype: object
Ano     int64
UF     object
dtype: object
Ano     int64
UF     object
dtype: object


In [ ]:
for df in [resultado, media_ideb, media_nivel_ide, media_grupos]:
    df['Ano'] = df['Ano'].astype(str)
    df['UF'] = df['UF'].astype(str)

In [ ]:
print(resultado[['UF', 'Ano']].drop_duplicates().shape)
print(media_ideb[['UF', 'Ano']].drop_duplicates().shape)
print(media_nivel_ide[['UF', 'Ano']].drop_duplicates().shape)
print(media_grupos[['UF', 'Ano']].drop_duplicates().shape)

(297, 2)
(243, 2)
(189, 2)
(135, 2)


In [ ]:
resultado['Ano'].unique()

array(['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2021', '2022', '2023'], dtype=object)

In [ ]:
media_ideb['Ano'].unique()

array(['2005', '2007', '2009', '2011', '2013', '2015', '2017', '2019',
       '2021'], dtype=object)

In [ ]:
media_nivel_ide['Ano'].unique()

array(['2013', '2014', '2015', '2016', '2017', '2018', '2019'],
      dtype=object)

In [ ]:
media_grupos['Ano'].unique()

array(['2013', '2014', '2015', '2016', '2017'], dtype=object)

In [ ]:
# Inicia com o df resultado
df_phi = resultado.merge(media_ideb, on=['Ano', 'UF'])
df_phi = df_phi.merge(media_nivel_ide, on=['Ano', 'UF'])
df_phi = df_phi.merge(media_grupos, on=['Ano', 'UF'])

In [ ]:
df_phi.columns

Index(['Ano', 'UF', 'Media_Ponderada', 'Media_IDEB', 'Nível IDE',
       'Media_Fundamental', 'Media_Medio'],
      dtype='object')

In [ ]:
df_phi['Soma_Total'] = (
    df_phi['Media_Ponderada'] +
    df_phi['Media_IDEB'] +
    df_phi['Nível IDE'] +
    df_phi['Media_Fundamental'] +
    df_phi['Media_Medio']
)

In [ ]:
df_phi['Soma_Total_ponderado'] = df_phi['Soma_Total']/5

In [ ]:
df_phi.head()

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
1,2013,AL,2.565217,3.800000,36.638196,27.289150,47.455682,117.748245,23.549649
2,2013,AM,2.944444,4.454545,44.677749,21.917828,56.723291,130.717859,26.143572
3,2013,AP,2.571429,4.036364,48.476338,28.557887,70.050806,153.692824,30.738565
4,2013,BA,2.886364,4.081818,36.122779,21.064921,31.996504,96.152386,19.230477


In [ ]:
df_phi['Ano'] = pd.to_numeric(df_phi['Ano'], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

ufs = df_phi['UF'].unique()

In [ ]:
# Lista para guardar as estimativas de 2012
estimados_2012 = []

for uf in ufs:
    df_uf = df_phi[(df_phi['UF'] == uf) & (df_phi['Ano'].isin([2013, 2015, 2017]))]

    if len(df_uf) == 3:
        # Ordenar por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajustar regressão linear
        model = LinearRegression().fit(X, y)

        # Prever o valor para 2012 (extrapolando para antes de 2013)
        valor_2012 = model.predict(np.array([[2012]]))[0]

        estimados_2012.append({'UF': uf, 'Ano': 2012, 'Soma_Total_ponderado': valor_2012})
    else:
        print(f"UF {uf} não tem exatamente 3 pontos de referência.")

In [ ]:
# Cria um DataFrame com as estimativas
df_estimados_2012 = pd.DataFrame(estimados_2012)

# Opcional: juntar com seu DataFrame original
df_final = pd.concat([df_phi, df_estimados_2012], ignore_index=True)

# Ordenar se desejar
df_final = df_final.sort_values(['UF', 'Ano'])

In [ ]:
df_final

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
81,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
0,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
27,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
54,2017,AC,3.000000,5.290909,54.310448,16.925302,30.145833,109.672492,21.934498
82,2012,AL,NaN,NaN,NaN,NaN,NaN,NaN,23.378583
...,...,...,...,...,...,...,...,...,...
79,2017,SP,3.086705,5.700000,13.393798,69.826962,63.453375,155.460841,31.092168
107,2012,TO,NaN,NaN,NaN,NaN,NaN,NaN,25.073013
26,2013,TO,2.450000,4.645455,37.274108,39.616795,41.670192,125.656550,25.131310
53,2015,TO,2.523810,4.709091,35.946091,40.041150,40.934824,124.154967,24.830993


In [ ]:
ufs = df_phi['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2015, 2017]

# Lista para armazenar as previsões de 2014
estimados_2014 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final[(df_final['UF'] == uf) & (df_final['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2014
        valor_2014 = model.predict(np.array([[2014]]))[0]

        estimados_2014.append({'UF': uf, 'Ano': 2014, 'Soma_Total_ponderado': valor_2014})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2014
df_estimados_2014 = pd.DataFrame(estimados_2014)

# Concatenar com os dados originais
df_final2 = pd.concat([df_final, df_estimados_2014], ignore_index=True)

# Ordenar
df_final2 = df_final2.sort_values(['UF', 'Ano'])

In [ ]:
df_final2.head()

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
1,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
108,2014,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.535074
2,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
3,2017,AC,3.000000,5.290909,54.310448,16.925302,30.145833,109.672492,21.934498


In [ ]:
ufs = df_final2['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2017]

# Lista para armazenar as previsões de 2014
estimados_2018 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final2[(df_final2['UF'] == uf) & (df_final2['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2018 = model.predict(np.array([[2018]]))[0]

        estimados_2018.append({'UF': uf, 'Ano': 2018, 'Soma_Total_ponderado': valor_2018})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2018
df_estimados_2018 = pd.DataFrame(estimados_2018)

# Concatenar com os dados originais
df_final3 = pd.concat([df_final2, df_estimados_2018], ignore_index=True)

# Ordenar
df_final3 = df_final3.sort_values(['UF', 'Ano'])

In [ ]:
df_final3.tail()

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
131,2013,TO,2.450000,4.645455,37.274108,39.616795,41.670192,125.656550,25.131310
132,2014,TO,NaN,NaN,NaN,NaN,NaN,NaN,25.022843
133,2015,TO,2.523810,4.709091,35.946091,40.041150,40.934824,124.154967,24.830993
134,2017,TO,2.666667,5.209091,36.536132,39.366492,41.376471,125.154852,25.030970
161,2018,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.922503


In [ ]:
ufs = df_final3['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2017, 2018]

# Lista para armazenar as previsões de 2014
estimados_2016 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final3[(df_final3['UF'] == uf) & (df_final3['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2016 = model.predict(np.array([[2016]]))[0]

        estimados_2016.append({'UF': uf, 'Ano': 2016, 'Soma_Total_ponderado': valor_2016})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2018
df_estimados_2016 = pd.DataFrame(estimados_2016)

# Concatenar com os dados originais
df_final4 = pd.concat([df_final3, df_estimados_2016], ignore_index=True)

# Ordenar
df_final4 = df_final4.sort_values(['UF', 'Ano'])

In [ ]:
df_final4.head()

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
1,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
2,2014,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.535074
3,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
162,2016,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.181957


In [ ]:
ufs = df_final4['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2016, 2017, 2018]

# Lista para armazenar as previsões de 2014
estimados_2019 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final4[(df_final4['UF'] == uf) & (df_final4['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2019 = model.predict(np.array([[2019]]))[0]

        estimados_2019.append({'UF': uf, 'Ano': 2019, 'Soma_Total_ponderado': valor_2019})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2019
df_estimados_2019 = pd.DataFrame(estimados_2019)

# Concatenar com os dados originais
df_final5 = pd.concat([df_final4, df_estimados_2019], ignore_index=True)

# Ordenar
df_final5 = df_final5.sort_values(['UF', 'Ano'])

In [ ]:
df_final5

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
1,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
2,2014,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.535074
3,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
4,2016,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.181957
...,...,...,...,...,...,...,...,...,...
185,2015,TO,2.523810,4.709091,35.946091,40.041150,40.934824,124.154967,24.830993
186,2016,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.972673
187,2017,TO,2.666667,5.209091,36.536132,39.366492,41.376471,125.154852,25.030970
188,2018,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.922503


In [ ]:
ufs = df_final5['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

# Lista para armazenar as previsões de 2014
estimados_2020 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final5[(df_final5['UF'] == uf) & (df_final5['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2020 = model.predict(np.array([[2020]]))[0]

        estimados_2020.append({'UF': uf, 'Ano': 2020, 'Soma_Total_ponderado': valor_2020})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2020
df_estimados_2020 = pd.DataFrame(estimados_2020)

# Concatenar com os dados originais
df_final6 = pd.concat([df_final5, df_estimados_2020], ignore_index=True)

# Ordenar
df_final6 = df_final6.sort_values(['UF', 'Ano'])

In [ ]:
df_final6.head()

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
1,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
2,2014,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.535074
3,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
4,2016,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.181957


In [ ]:
ufs = df_final6['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

# Lista para armazenar as previsões de 2014
estimados_2021 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final6[(df_final6['UF'] == uf) & (df_final6['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2021 = model.predict(np.array([[2021]]))[0]

        estimados_2021.append({'UF': uf, 'Ano': 2021, 'Soma_Total_ponderado': valor_2021})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2020
df_estimados_2021 = pd.DataFrame(estimados_2021)

# Concatenar com os dados originais
df_final7 = pd.concat([df_final6, df_estimados_2021], ignore_index=True)

# Ordenar
df_final7 = df_final7.sort_values(['UF', 'Ano'])

In [ ]:
df_final7

,Ano,UF,Media_Ponderada,Media_IDEB,Nível IDE,Media_Fundamental,Media_Medio,Soma_Total,Soma_Total_ponderado
0,2012,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.888191
1,2013,AC,2.625000,4.800000,56.129086,15.833823,33.815758,113.203666,22.640733
2,2014,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.535074
3,2015,AC,2.818182,4.990909,56.885832,15.890691,31.915957,112.501571,22.500314
4,2016,AC,NaN,NaN,NaN,NaN,NaN,NaN,22.181957
...,...,...,...,...,...,...,...,...,...
239,2017,TO,2.666667,5.209091,36.536132,39.366492,41.376471,125.154852,25.030970
240,2018,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.922503
241,2019,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.897418
242,2020,TO,NaN,NaN,NaN,NaN,NaN,NaN,24.872333


In [ ]:
ufs = df_final7['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

# Lista para armazenar as previsões de 2014
estimados_2022 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final7[(df_final7['UF'] == uf) & (df_final7['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2022 = model.predict(np.array([[2022]]))[0]

        estimados_2022.append({'UF': uf, 'Ano': 2022, 'Soma_Total_ponderado': valor_2022})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2020
df_estimados_2022 = pd.DataFrame(estimados_2022)

# Concatenar com os dados originais
df_final8 = pd.concat([df_final7, df_estimados_2022], ignore_index=True)

# Ordenar
df_final8 = df_final8.sort_values(['UF', 'Ano'])

In [ ]:
ufs = df_final8['UF'].unique()

# Lista de anos de referência para prever 2014
anos_referencia = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

# Lista para armazenar as previsões de 2014
estimados_2023 = []

for uf in ufs:
    # Pega os valores de 2012, 2013, 2015 e 2017
    df_uf = df_final8[(df_final8['UF'] == uf) & (df_final8['Ano'].isin(anos_referencia))]

    if len(df_uf) >= 2:
        # Ordena por ano
        df_uf = df_uf.sort_values('Ano')
        X = np.array(df_uf['Ano']).reshape(-1, 1)
        y = df_uf['Soma_Total_ponderado'].values

        # Ajusta a regressão linear
        model = LinearRegression().fit(X, y)

        # Previsão para 2018
        valor_2023 = model.predict(np.array([[2023]]))[0]

        estimados_2023.append({'UF': uf, 'Ano': 2023, 'Soma_Total_ponderado': valor_2023})
    else:
        print(f"UF {uf} não tem pontos suficientes para ajuste (precisa de pelo menos 2).")

# DataFrame com previsões de 2020
df_estimados_2023 = pd.DataFrame(estimados_2023)

# Concatenar com os dados originais
df_final9 = pd.concat([df_final8, df_estimados_2023], ignore_index=True)

# Ordenar
df_final9 = df_final9.sort_values(['UF', 'Ano'])

In [ ]:
df_final9.to_csv('df_final9.csv')